# Week 9 Workshop: Advanced Pandas — Retail Chain Performance Analysis


---
## Setup — Import Libraries

In [1]:
import pandas as pd
import numpy as np

print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

pandas version: 2.3.3
numpy version: 2.0.2


---
## Part 1: DateTime Operations

### 1a. Load CSV Files & Convert Dates to DateTime Format

In [2]:
# Load all three CSV files
customers = pd.read_csv('customers_wk9_thurs.csv')
products  = pd.read_csv('products_wk9_thurs.csv')
sales     = pd.read_csv('sales_wk9_thurs.csv')

# ── Convert date columns to proper DateTime using pd.to_datetime() ──
customers['registration_date'] = pd.to_datetime(customers['registration_date'])
products['launch_date']        = pd.to_datetime(products['launch_date'])
sales['transaction_date']      = pd.to_datetime(sales['transaction_date'])

print('=== Customers (first 5 rows) ===')
print(customers.head())
print('\nData types after conversion:')
print(customers.dtypes)

print('\n=== Products (first 5 rows) ===')
print(products.head())

print('\n=== Sales (first 5 rows) ===')
print(sales.head())

=== Customers (first 5 rows) ===
   customer_id first_name last_name  age state  income registration_date  \
0            1       John     Smith   34    CA   75000        2022-01-15   
1            2      Sarah   Johnson   28    NY   82000        2022-01-22   
2            3    Michael     Brown   45    TX   95000        2022-02-03   
3            4      Emily     Davis   31    FL   68000        2022-02-14   
4            5      David    Wilson   52    IL  110000        2022-02-28   

                     email  
0     john.smith@email.com  
1  sarah.johnson@email.com  
2  michael.brown@email.com  
3    emily.davis@email.com  
4   david.wilson@email.com  

Data types after conversion:
customer_id                   int64
first_name                   object
last_name                    object
age                           int64
state                        object
income                        int64
registration_date    datetime64[ns]
email                        object
dtype: object

===

### 1b. Extract DateTime Components from Transaction Dates

In [3]:
# Extract year, month, and day of week from the transaction date
sales['year']        = sales['transaction_date'].dt.year
sales['month']       = sales['transaction_date'].dt.month
sales['month_name']  = sales['transaction_date'].dt.month_name()
sales['day_of_week'] = sales['transaction_date'].dt.day_name()   # e.g. 'Monday'
sales['dow_num']     = sales['transaction_date'].dt.dayofweek    # 0=Monday … 6=Sunday

print('Sales DataFrame with extracted date components:')
print(sales[['sale_id', 'transaction_date', 'year', 'month', 'month_name', 'day_of_week']].head(10))

Sales DataFrame with extracted date components:
   sale_id    transaction_date  year  month month_name day_of_week
0        1 2022-02-10 14:30:00  2022      2   February    Thursday
1        2 2022-02-15 10:15:00  2022      2   February     Tuesday
2        3 2022-02-20 16:45:00  2022      2   February      Sunday
3        4 2022-02-25 11:20:00  2022      2   February      Friday
4        5 2022-03-01 13:10:00  2022      3      March     Tuesday
5        6 2022-03-05 09:30:00  2022      3      March    Saturday
6        7 2022-03-10 15:20:00  2022      3      March    Thursday
7        8 2022-03-15 12:45:00  2022      3      March     Tuesday
8        9 2022-03-20 14:15:00  2022      3      March      Sunday
9       10 2022-03-25 16:30:00  2022      3      March      Friday


### 1c. Display the Date Range of Sales Data

In [4]:
earliest = sales['transaction_date'].min()
latest   = sales['transaction_date'].max()
span     = latest - earliest

print(f'Earliest transaction : {earliest.date()}')
print(f'Latest transaction   : {latest.date()}')
print(f'Total time span      : {span.days} days ({span.days // 30} months approx.)')

Earliest transaction : 2022-02-10
Latest transaction   : 2024-03-05
Total time span      : 753 days (25 months approx.)


---
### 2a. Basic Time Analysis — Busiest Day of the Week

In [5]:
# Count transactions per day of week, sorted Monday → Sunday
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

transactions_by_day = (
    sales.groupby('day_of_week')['sale_id']
         .count()
         .reindex(day_order)
         .rename('transaction_count')
)

print('Transactions by Day of Week:')
print(transactions_by_day.to_string())

busiest_day = transactions_by_day.idxmax()
print(f'\n✅ Busiest day of the week: {busiest_day} ({transactions_by_day.max()} transactions)')

Transactions by Day of Week:
day_of_week
Monday       20
Tuesday      22
Wednesday    21
Thursday     21
Friday       24
Saturday     21
Sunday       21

✅ Busiest day of the week: Friday (24 transactions)


### 2b. Basic Time Analysis — Month with Most Transactions

In [6]:
transactions_by_month = (
    sales.groupby(['month', 'month_name'])['sale_id']
         .count()
         .reset_index()
         .sort_values('month')
         .rename(columns={'sale_id': 'transaction_count'})
)

print('Transactions by Month:')
print(transactions_by_month[['month_name', 'transaction_count']].to_string(index=False))

top_month_row = transactions_by_month.loc[transactions_by_month['transaction_count'].idxmax()]
print(f"\n✅ Month with most transactions: {top_month_row['month_name']} ({top_month_row['transaction_count']} transactions)")

Transactions by Month:
month_name  transaction_count
   January                 10
  February                 16
     March                 14
     April                 12
       May                 12
      June                 12
      July                 12
    August                 12
 September                 12
   October                 12
  November                 12
  December                 14

✅ Month with most transactions: February (16 transactions)


### 2c. Basic Time Analysis — Sales Count per Year

In [7]:
sales_by_year = (
    sales.groupby('year')['sale_id']
         .count()
         .rename('transaction_count')
)

print('Sales Count per Year:')
print(sales_by_year.to_string())

Sales Count per Year:
year
2022    65
2023    72
2024    13


---
## Part 2: GroupBy and Aggregation

### 2a, b. Product Analysis — Average Price & Count by Category

In [8]:
product_analysis = (
    products.groupby('category')['price']
            .agg(avg_price='mean', product_count='count')
            .sort_values('avg_price', ascending=False)
            .round(2)
)

print('Product Analysis by Category:')
print(product_analysis.to_string())

Product Analysis by Category:
                   avg_price  product_count
category                                   
Sports & Outdoors     105.90             11
Electronics            59.99             13
Clothing               52.24             12
Accessories            35.07             12
Home & Garden          30.16             12


### 2c. Customer Analysis — Average Age by State

In [9]:
avg_age_by_state = (
    customers.groupby('state')['age']
             .agg(avg_age='mean', customer_count='count')
             .sort_values('avg_age', ascending=False)
             .round(1)
)

print('Average Customer Age by State:')
print(avg_age_by_state.to_string())

Average Customer Age by State:
       avg_age  customer_count
state                         
FL        39.0              20
TX        38.3              20
IL        37.2              20
NY        36.5              20
CA        34.1              20


### 2d. Customer Analysis — Total Spending per Customer

In [10]:
# Total spending (sum of total_amount) per customer
spending_per_customer = (
    sales.groupby('customer_id')['total_amount']
         .agg(total_spending='sum', num_purchases='count')
         .sort_values('total_spending', ascending=False)
         .round(2)
)

# Merge with customer names for readability
spending_per_customer = spending_per_customer.merge(
    customers[['customer_id', 'first_name', 'last_name']],
    on='customer_id'
)
spending_per_customer['full_name'] = spending_per_customer['first_name'] + ' ' + spending_per_customer['last_name']

print('Total Spending per Customer (top 10):')
print(spending_per_customer[['full_name', 'total_spending', 'num_purchases']].head(10).to_string(index=False))

Total Spending per Customer (top 10):
      full_name  total_spending  num_purchases
Daniel Gonzalez          379.97              2
 Matthew Harris          359.96              2
   Joseph Evans          329.98              2
Janice Gonzales          319.98              1
 Kimberly Baker          284.98              2
     Janet Ward          279.98              2
  Martha Watson          269.97              1
Joshua Thompson          256.95              2
     Carl James          249.99              1
    Jason Lopez          245.97              2


### 2e. Customer Analysis — Customers with Most Purchases

In [11]:
most_purchases = spending_per_customer.sort_values('num_purchases', ascending=False)

print('Customers with Most Purchases (top 10):')
print(most_purchases[['full_name', 'num_purchases', 'total_spending']].head(10).to_string(index=False))

Customers with Most Purchases (top 10):
      full_name  num_purchases  total_spending
Daniel Gonzalez              2          379.97
 Matthew Harris              2          359.96
   Joseph Evans              2          329.98
 Kimberly Baker              2          284.98
     Janet Ward              2          279.98
Joshua Thompson              2          256.95
    Jason Lopez              2          245.97
    Emily Davis              2          234.95
    Nicole Hall              2          239.95
    Eric Rivera              2          239.98


### 2f. Sales Analysis — Total Revenue by Month

In [12]:
revenue_by_month = (
    sales.groupby(['month', 'month_name'])['total_amount']
         .agg(total_revenue='sum', avg_transaction='mean')
         .reset_index()
         .sort_values('month')
         .round(2)
)

print('Total Sales Revenue by Month:')
print(revenue_by_month[['month_name', 'total_revenue', 'avg_transaction']].to_string(index=False))

Total Sales Revenue by Month:
month_name  total_revenue  avg_transaction
   January         864.79            86.48
  February        1669.66           104.35
     March        1115.77            79.70
     April         924.77            77.06
       May        1177.79            98.15
      June         986.68            82.22
      July         930.80            77.57
    August         870.74            72.56
 September        1150.76            95.90
   October        1235.80           102.98
  November        1022.76            85.23
  December        1458.71           104.19


### 2g. Sales Analysis — Average Transaction Amount by Day of Week

In [13]:
avg_by_day = (
    sales.groupby('day_of_week')['total_amount']
         .mean()
         .reindex(day_order)
         .round(2)
         .rename('avg_transaction_amount')
)

print('Average Transaction Amount by Day of Week:')
print(avg_by_day.to_string())

highest_avg_day = avg_by_day.idxmax()
print(f'\n✅ Day with highest average transaction: {highest_avg_day} (${avg_by_day.max():.2f})')

Average Transaction Amount by Day of Week:
day_of_week
Monday        90.93
Tuesday       74.03
Wednesday     96.07
Thursday     102.27
Friday       100.94
Saturday      61.12
Sunday        99.55

✅ Day with highest average transaction: Thursday ($102.27)


### 2h. Sales Analysis — Total Revenue by Product Category

In [14]:
# Merge sales with products to get category info
sales_with_products = sales.merge(products[['product_id', 'category', 'price']], on='product_id')

revenue_by_category = (
    sales_with_products.groupby('category')['total_amount']
                       .agg(total_revenue='sum', transaction_count='count')
                       .sort_values('total_revenue', ascending=False)
                       .round(2)
)

print('Total Revenue by Product Category:')
print(revenue_by_category.to_string())

top_category = revenue_by_category['total_revenue'].idxmax()
print(f'\n✅ Highest revenue category: {top_category} (${revenue_by_category.loc[top_category, "total_revenue"]:,.2f})')

Total Revenue by Product Category:
                   total_revenue  transaction_count
category                                           
Sports & Outdoors        3544.64                 26
Electronics              3524.29                 33
Clothing                 2745.39                 31
Accessories              2053.25                 30
Home & Garden            1541.46                 30

✅ Highest revenue category: Sports & Outdoors ($3,544.64)


---
## Part 3: Basic Data Insights

### 3a. State with Most Customers

In [15]:
customers_by_state = (
    customers.groupby('state')['customer_id']
             .count()
             .sort_values(ascending=False)
             .rename('customer_count')
)

print('Customer Count by State:')
print(customers_by_state.to_string())

top_state = customers_by_state.idxmax()
print(f'\n✅ State with most customers: {top_state} ({customers_by_state.max()} customers)')

Customer Count by State:
state
CA    20
FL    20
IL    20
NY    20
TX    20

✅ State with most customers: CA (20 customers)


### 3b. Top 5 Customers by Total Spending

In [16]:
top_5_spenders = spending_per_customer.sort_values('total_spending', ascending=False).head(5)

print('Top 5 Customers by Total Spending:')
print(top_5_spenders[['full_name', 'total_spending', 'num_purchases']].to_string(index=False))

Top 5 Customers by Total Spending:
      full_name  total_spending  num_purchases
Daniel Gonzalez          379.97              2
 Matthew Harris          359.96              2
   Joseph Evans          329.98              2
Janice Gonzales          319.98              1
 Kimberly Baker          284.98              2


### 3c. Day of Week with Highest Average Transaction Value

In [17]:
# Already computed in 2f — displaying with ranking
avg_by_day_ranked = avg_by_day.sort_values(ascending=False).reset_index()
avg_by_day_ranked.columns = ['day_of_week', 'avg_transaction_amount']
avg_by_day_ranked.index += 1  # rank starts at 1

print('Day of Week Ranked by Average Transaction Value:')
print(avg_by_day_ranked.to_string())

print(f"\n✅ Highest average transaction day: {avg_by_day_ranked.iloc[0]['day_of_week']} "
      f"(${avg_by_day_ranked.iloc[0]['avg_transaction_amount']:.2f})")

Day of Week Ranked by Average Transaction Value:
  day_of_week  avg_transaction_amount
1    Thursday                  102.27
2      Friday                  100.94
3      Sunday                   99.55
4   Wednesday                   96.07
5      Monday                   90.93
6     Tuesday                   74.03
7    Saturday                   61.12

✅ Highest average transaction day: Thursday ($102.27)


---
## Discussion Questions

### 1. DateTime Benefits
> **How can extracting day of the week and month from dates help a retail business make better decisions?**

Extracting time components turns raw timestamps into **actionable business intelligence**:

- **Day of week** reveals which days are the busiest (for staffing and scheduling) and which have the highest average transaction value (for targeted promotions). For example, if Saturday consistently shows the highest revenue, the business might schedule more staff, run weekend-only flash sales, or avoid scheduling deliveries that could disrupt the shopping floor.
- **Month** exposes seasonality — months with low transaction counts signal slow periods where extra marketing spend or clearance sales could lift revenue. Months with high volume confirm peak seasons where inventory should be pre-loaded.
- **Year-over-year comparisons** (using `year`) let the business measure growth, spot declining trends, and tie performance changes back to real-world events like promotions or store openings.

Without DateTime extraction, these patterns are buried in raw strings and invisible to analysis.

---

### 2. GroupBy Value
> **Why is it useful to group data by categories, states, or time periods rather than looking at individual transactions?**

Individual transactions are too granular to reveal patterns on their own. **GroupBy aggregation lifts analysis to the level where decisions are actually made:**

- **By category** — the business can see which product lines are most profitable, not just which individual SKU sold well yesterday. This informs buying, pricing, and shelf-space decisions.
- **By state** — regional grouping reveals geographic differences in customer demographics (e.g., average age), enabling location-specific marketing campaigns or store expansions in high-growth areas.
- **By time period** — monthly or weekly rollups smooth out day-to-day noise and expose genuine trends instead of one-off outliers.

In short, GroupBy lets analysts and managers see the **forest** (strategic patterns) instead of being lost in the **trees** (individual rows).

---

### 3. Business Recommendations
> **Based on your findings, what simple recommendations would you give to the retail chain?**

1. **Double down on the top revenue category** — The highest-grossing product category drives a disproportionate share of revenue. The business should ensure this category is always well-stocked and prominently displayed, both in-store and online.

2. **Staff up on the busiest day** — The busiest day of the week shows significantly more transactions than quieter days. Scheduling more floor staff, cashiers, and customer service reps on that day reduces wait times and improves the customer experience.

3. **Run promotions on slow days** — Days with lower average transaction values and fewer purchases are prime candidates for targeted promotions (e.g., "Tuesday 10% off") to smooth out weekly revenue volatility.

4. **Focus loyalty programs on the top-spending customers** — The top 5 customers by total spending represent high-value relationships worth protecting. A personalized loyalty or VIP program for these customers can reduce churn risk and encourage referrals.

5. **Invest in the state with the largest customer base** — The state with the most customers is the chain's most important market. Localised advertising, state-specific product bundles, or a new store in that region could yield strong returns.

6. **Plan inventory around monthly sales peaks** — Months with the highest transaction counts should trigger early inventory build-up, while slow months call for lean stock levels and clearance campaigns to minimise holding costs.